Task 1: Load and inspect the dataset

In [1]:
import pandas as pd
scores = pd.read_csv("data_safe_copy.csv")

In [2]:
assert len(scores) >= 300

In [3]:
scores.head()

,student_id,cohort,module,assignment,score
0,S001,alpha,Module1,A1,78
1,S001,alpha,Module1,A2,84
2,S001,alpha,Module2,A1,79
3,S001,alpha,Module2,A2,86
4,S001,alpha,Module3,A1,81


In [4]:
scores.info()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   student_id  300 non-null    str  
 1   cohort      300 non-null    str  
 2   module      300 non-null    str  
 3   assignment  300 non-null    str  
 4   score       300 non-null    int64
dtypes: int64(1), str(4)
memory usage: 11.8 KB


In [5]:
assert scores["score"].apply(lambda x: isinstance(x, (int, float))).all()

Task 2: Build a student roster table and merge

In [6]:
student_id = [
    "S001", "S003", "S005", "S007", "S009",
    "S011", "S013", "S015", "S017", "S019", 
    "S350", "S351", "S352", "S353", "S354",
    "S355", "S356", "S357", "S358", "S359"  
]

status = [
    "active", "inactive", "active", "inactive", "active",
    "active", "inactive", "inactive", "active", "inactive",
    "inactive", "active", "inactive", "active", "inactive",
    "active", "inactive", "active", "inactive", "active"
]

roster = pd.DataFrame({
    "student_id": student_id,
    "status": status
})


In [7]:
merged = scores.merge(roster, on="student_id", how="left")
missing_status = merged["status"].isna().sum()
print("Missing statuses:", missing_status)
assert missing_status > 0
assert len(merged) == len(scores)

Missing statuses: 240


Task 3: Aggregate by module and cohort

In [9]:
avg_score_by_module = scores.groupby(["cohort", "module"])["score"].mean() 
avg_by_module = pd.DataFrame({
    "avg_score_by_module": avg_score_by_module
})

In [10]:
avg_by_module

avg_score_by_module
cohort module                      
alpha  Module1            77.970588
       Module2            78.382353
       Module3            79.794118
beta   Module1            78.647059
       Module2            78.411765
       Module3            80.029412
gamma  Module1            76.218750
       Module2            76.468750
       Module3            77.843750

In [11]:
import numpy as np
avg_score_by_cohort = scores.groupby("cohort")["score"].mean() 
rollup_avg = avg_by_module.groupby(level=0)["avg_score_by_module"].mean()
print("Rolled-up module averages by cohort:")
print(rollup_avg)

print("\nOverall average by cohort:")
print(avg_score_by_cohort)

comparison = np.isclose(rollup_avg, avg_score_by_cohort)
print("\nDo rolled-up module averages match cohort-level averages?", comparison)

Rolled-up module averages by cohort:
cohort
alpha    78.715686
beta     79.029412
gamma    76.843750
Name: avg_score_by_module, dtype: float64

Overall average by cohort:
cohort
alpha    78.715686
beta     79.029412
gamma    76.843750
Name: score, dtype: float64

Do rolled-up module averages match cohort-level averages? [ True  True  True]


Task 4: Reshape to a wide report

In [30]:
student_module_report = scores.pivot_table(index="student_id", columns="module", values="score", aggfunc="mean")
student_module_report.head()

module,Module1,Module2,Module3
student_id,,,
S001,81.0,82.5,84.5
S002,73.5,72.0,75.0
S003,87.5,88.5,89.5
S004,68.5,67.5,69.5
S005,81.5,82.5,83.5


In [31]:
assert student_module_report.shape[0] == scores['student_id'].nunique()

Task 5: Ranking and top performers

In [45]:
avg_scores = scores.groupby(['student_id', 'cohort'], as_index=False)['score'].mean()
avg_scores.rename(columns={'score': 'avg_score'}, inplace=True)

top_students = avg_scores.sort_values(['cohort', 'avg_score'], ascending=[True, False]) \
                         .groupby('cohort') \
                         .head(3)

print(top_students)

assert all(top_students['cohort'].value_counts() == 3)

   student_id cohort  avg_score
2        S003  alpha       88.5
48       S049  alpha       88.5
26       S027  alpha       86.5
44       S045   beta       90.5
6        S007   beta       89.5
20       S021   beta       88.5
13       S014  gamma       92.5
39       S040  gamma       84.5
15       S016  gamma       84.0
